## Transformers, Quantization, and Neural Networks

In [ ]:
!pip install -q --upgrade bitsandbytes accelerate transformers==4.57.6

In [ ]:
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokeniser, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gc

In [ ]:
hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)


In [ ]:
LLAMA = 'meta-llama/Llama-3.1-8B'
PHI = 'microsoft/Phi-4-mini-instruct'
GEMMA = 'google/gemma-3-270m-it'
DEEPSEEK = 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B'


In [ ]:
messages = [
    {"role":"system", "content":" You are a helpful assistant"},
    {"role":"user", "content": "Tell a joke for a room of Physicists"}
]

In [ ]:
# Quantization Config - this allows us to load the model into memory and use less memory
quant_config = BitsAndBytesConfig(
    load_in_4bit=True, #load weights in 4 bit precision (less memory)
    bnb_4bit_use_double_quant=True, #weights and the quantization constants are quatized (reduces memory usage further)
    bnb_4bit_compute_dtype=torch.bfloat16, #sets computation data type
    #bfloat16 is fast, stable and supported on many modern gpus
    bnb_4bit_quant_type='nf4' #special 4 bit format for neural networks
)

In [ ]:
from transformers import AutoTokenizer


tokenizer = AutoTokenizer.from_pretrained(LLAMA) #use other models if restricted
tokenizer.pad_token = tokenizer.eos_token #some models need this to be set as 'end of sentence' token
inputs = tokenizer.apply_chat_template(messages, return_tensors='pt').to("cuda")

In [ ]:
inputs

In [ ]:
#the model
model = AutoModelForCausalLM.from_pretrained(QWEN, device_map='auto', quantization_config=quant_config)

# causalLM/auto-regressor = LLMs that generate content

In [ ]:
# view models memory footprint

memory = model.get_memory_footprint()/ 1e6
print(f"Memory footprint: {memory:,.1f}MB")

The next cell prints the HuggingFace model object for Llama.

This model object is a Neural Network, implemented with the Python framework PyTorch. The Neural Network uses the architecture invented by Google scientists in 2017: the Transformer architecture.

While we're not going to go deep into the theory, this is an opportunity to get some intuition for what the Transformer actually is.

If you're completely new to Neural Networks, check out my YouTube intro playlist for the foundations.

Now take a look at the layers of the Neural Network that get printed in the next cell. Look out for this:

It consists of layers
There's something called "embedding" - this takes tokens and turns them into 4,096 dimensional vectors. We'll learn more about this in Week 5.
There are then 16 sets of groups of layers (32 for Llama 3.1) called "Decoder layers". Each Decoder layer contains three types of layer: (a) self-attention layers (b) multi-layer perceptron (MLP) layers (c) batch norm layers.
There is an LM Head layer at the end; this produces the output
Notice the mention that the model has been quantized to 4 bits.

It's not required to go any deeper into the theory at this point, but if you'd like to, I've asked our mutual friend to take this printout and make a tutorial to walk through each layer. This also looks at the dimensions at each point. If you're interested, work through this tutorial after running the next cell:

https://chatgpt.com/canvas/shared/680cbea6de688191a20f350a2293c76b

In [ ]:
model


In addition to looking at each of the layers in the model, you can actually look at the HuggingFace code that implements Llama using PyTorch.

Here is the HuggingFace Transformers repo:
https://github.com/huggingface/transformers

And within this, here is the code for Llama 4:
https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama4/modeling_llama4.py

Obviously it's not neceesary at all to get into this detail - the job of an AI engineer is to select, optimize, fine-tune and apply LLMs rather than to code a transformer in PyTorch. OpenAI, Meta and other frontier labs spent millions building and training these models. But it's a fascinating rabbit hole if you're interested!

In [ ]:
# run model

outputs = model.generate(inputs, max_new_tokens=80)
outputs[0] #output = tokens

In [ ]:
tokenizer.decode(outputs[0])

In [ ]:
# Clean up memory
# Thank you Kuan L. for helping me get this to properly free up memory!
# If you select "Show Resources" on the top right to see GPU memory, it might not drop down right away
# But it does seem that the memory is available for use by new models in the later code.

del model, inputs, tokenizer, outputs
gc.collect()
torch.cuda.empty_cache()

A couple of quick notes on the next block of code:

I'm using a HuggingFace utility called TextStreamer so that results stream back. To stream results, we simply replace:
outputs = model.generate(inputs, max_new_tokens=80)
With:
streamer = TextStreamer(tokenizer)
outputs = model.generate(inputs, max_new_tokens=80, streamer=streamer)

Also I've added the argument add_generation_prompt=True to my call to create the Chat template. This ensures that Phi generates a response to the question, instead of just predicting how the user prompt continues. Try experimenting with setting this to False to see what happens. You can read about this argument here:

https://huggingface.co/docs/transformers/main/en/chat_templating#what-are-generation-prompts

Thank you to student Piotr B for raising the issue!

In [ ]:
# Wrapping everything in a function - and adding Streaming and generation prompts

def generate(model, messages, quant=True, max_new_tokens=80):
  tokenizer = AutoTokenizer.from_pretrained(model)
  tokenizer.pad_token = tokenizer.eos_token
  input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True).to("cuda")
  attention_mask = torch.ones_like(input_ids, dtype=torch.long, device="cuda")
  streamer = TextStreamer(tokenizer)
  if quant:
    model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
  else:
    model = AutoModelForCausalLM.from_pretrained(model).to("cuda")
  outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_new_tokens=max_new_tokens, streamer=streamer)



In [ ]:
messages = [
    {"role": "user", "content": "Tell a light-hearted joke for a room of Data Scientists"}
  ]
generate(GEMMA, messages, quant=False)

In [ ]:
generate(QWEN, messages)

In [ ]:
generate(DEEPSEEK, messages, quant=False, max_new_tokens=500)